In [1]:
import ee
ee.Authenticate()


Successfully saved authorization token.


In [2]:
ee.Initialize()

In [3]:
import geopandas as gpd

In [4]:
embeddings = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

In [5]:
from pathlib import Path

states_path = Path("data/raw/tl_2022_us_state/tl_2022_us_state.shp")               
states = gpd.read_file(states_path)
# TIGER state FIPS for Florida is 12
fl_gdf = states.loc[states["STATEFP"] == "12"].copy()
fl_gdf = fl_gdf.to_crs("EPSG:4326")
florida_geojson = fl_gdf.geometry.iloc[0].__geo_interface__
florida = ee.Geometry(florida_geojson)

DataSourceError: data/raw/tl_2022_us_state/tl_2022_us_state.shp: No such file or directory

In [ ]:
year = 2023                                                  
embedding = (
    embeddings
    .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
    .filterBounds(florida)                                          
    .mosaic()
    .clip(florida)
)

embedding.bandNames().getInfo()

['A00',
 'A01',
 'A02',
 'A03',
 'A04',
 'A05',
 'A06',
 'A07',
 'A08',
 'A09',
 'A10',
 'A11',
 'A12',
 'A13',
 'A14',
 'A15',
 'A16',
 'A17',
 'A18',
 'A19',
 'A20',
 'A21',
 'A22',
 'A23',
 'A24',
 'A25',
 'A26',
 'A27',
 'A28',
 'A29',
 'A30',
 'A31',
 'A32',
 'A33',
 'A34',
 'A35',
 'A36',
 'A37',
 'A38',
 'A39',
 'A40',
 'A41',
 'A42',
 'A43',
 'A44',
 'A45',
 'A46',
 'A47',
 'A48',
 'A49',
 'A50',
 'A51',
 'A52',
 'A53',
 'A54',
 'A55',
 'A56',
 'A57',
 'A58',
 'A59',
 'A60',
 'A61',
 'A62',
 'A63']

In [ ]:
# example Florida point, mature stand in the Ocala
# reference_point = ee.Geometry.Point([-81.8039417, 29.26155])  

# example Florda point #2, a clearcut stand in the Ocala
reference_point = ee.Geometry.Point([-81.8048667, 29.256675])

In [ ]:
bands = embedding.bandNames()

reference_vector = embedding.sample(
    region=reference_point,                                         
    scale=10,
    numPixels=1,
    geometries=True,
).first()

reference_dict = reference_vector.toDictionary(bands)

reference_dict.getInfo()

{'A00': -0.0629911572472126,
 'A01': -0.15378700499807765,
 'A02': -0.09356401384083046,
 'A03': 0.2364628988850442,
 'A04': -0.16000000000000003,
 'A05': -0.09356401384083046,
 'A06': 0.002214532871972318,
 'A07': -0.0629911572472126,
 'A08': 0.1085121107266436,
 'A09': 0.0629911572472126,
 'A10': -0.09842368319876971,
 'A11': -0.08421376393694734,
 'A12': -0.16000000000000003,
 'A13': 0.019930795847750864,
 'A14': -0.07535563244905807,
 'A15': 0.1085121107266436,
 'A16': -0.13588619761630144,
 'A17': 0.17937716262975778,
 'A18': 0.0629911572472126,
 'A19': -0.01384083044982699,
 'A20': -0.11909265667051133,
 'A21': 0.11374086889657825,
 'A22': -0.008858131487889272,
 'A23': 0.003936947327950788,
 'A24': -0.024605920799692427,
 'A25': 0.07972318339100345,
 'A26': -0.03844675124951941,
 'A27': -0.08882737408688965,
 'A28': -0.01384083044982699,
 'A29': 0.22145328719723184,
 'A30': 0.2288965782391388,
 'A31': 0.012056901191849288,
 'A32': -0.05536332179930796,
 'A33': 0.0517339484813533

In [ ]:
reference_image = ee.Image.constant(reference_dict.values(bands)).rename(bands)

In [ ]:
similarity = (                                                      
    embedding
    .multiply(reference_image)                                      
    .reduce(ee.Reducer.sum())
    .rename("similarity")
)

In [ ]:
# Optional similarity theshold masking
similar_pixels = similarity.updateMask(similarity.gt(0.85))

In [ ]:
'''
task = ee.batch.Export.image.toDrive(                               
    image=similarity,
    description=f"alphaearth_similarity_{year}",
    folder="artemis_exports",
    fileNamePrefix=f"alphaearth_similarity_{year}",                 
    region=florida,
    scale=10,
    crs="EPSG:5070",
    maxPixels=1e13,
)

task.start()
task.status()
'''

'\ntask = ee.batch.Export.image.toDrive(                               \n    image=similarity,\n    description=f"alphaearth_similarity_{year}",\n    folder="artemis_exports",\n    fileNamePrefix=f"alphaearth_similarity_{year}",                 \n    region=florida,\n    scale=10,\n    crs="EPSG:5070",\n    maxPixels=1e13,\n)\n\ntask.start()\ntask.status()\n'

In [ ]:
import geemap                                                
m = geemap.Map()
m.centerObject(reference_point, 10)
m.add_basemap("SATELLITE")
m.addLayer(
    similarity,
    {"min": 0.6, "max": 1.0, "palette": ["black", "blue",     "cyan", "yellow", "red"]},
    "AlphaEarth similarity",
)

m.addLayer(reference_point, {"color": "white"}, "Reference point")

m

Map(center=[29.256675, -81.8048667], controls=(WidgetControl(options=['position', 'transparent_bg'], position=…